# 09 · Горизонты прогноза: 6 / 12 / 24 / 48 часов

**Цель** — оценить, как падает качество прогноза при удлинении горизонта:
неисправность оборудования (насосы/вентиляторы/фазы) в течение
ближайших **6 / 12 / 24 / 48 часов**.

Подход:
- **6-часовой панэль** «канал × бакет» (`features.make_subdaily_panel`): события
  журнала СМВУ агрегируются в 6-часовые бакеты с учётом колонки `время`;
- признаки: счётчики за бакет + **календарные окна 12ч/24ч/48ч/7д/30д**
  (суммы по бакетам через префиксные суммы + searchsorted), z-скор событий,
  сезонность (час_бакета, день недели, месяц), **бакетный контекст объекта**
  (все каналы объекта за тот же 6ч интервал и его окна — `_об_б`);
- цели: `цель_6ч/12ч/24ч/48ч` = неисправность канала в течение следующих
  1/2/4/8 бакетов (NaN в кампанийных неделях);
- временной сплит: train ≤ 2024, val = 2025, test = 2026 (без кампаний);
- модель: CatBoost (CPU), оптимизация F1, early stopping по валидации;
- метрики: PR-AUC, Precision@Recall≥0.5, F1@0.5, порог под Precision≥0.7.

> Данные построены пайплайном `notebooks/00_data_pipeline.ipynb`
> → `dataset/subdaily_panel_wear_6h.csv` (шаги 2–3).

In [ ]:
import sys
import pathlib

_HERE = pathlib.Path.cwd()
if _HERE.name == "notebooks":
    RESEARCH = _HERE.parent
else:
    RESEARCH = _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data_utils as du
import features as fe

RECOMPUTE = False      # False — читать готовый суб-суточный панэль
print("dataset:", du.find_dataset_dir())

# 9.1 · Данные: 6-часовой панэль

Читаем `dataset/subdaily_panel_wear_6h.csv` (1.6M строк, 63 колонки).
Доля положительных растёт с горизонтом: 6ч → 12ч → 24ч → 48ч
(чем больше окно, тем вероятнее попасть в неисправность).

In [ ]:
panel = fe.make_subdaily_panel(recompute=RECOMPUTE)
print("Суб-суточный панэль:", panel.shape)
print("Колонки:", list(panel.columns))
print("\nДоля целевых переменных (чистые дни, без кампаний):")
for name in fe.HORIZONS:
    y = panel[f"цель_{name}"].dropna()
    print(f"  цель_{name}: {100 * y.mean():6.3f}% положительных  (n={len(y):,})")
panel.head()

# 9.2 · Временной сплит и признаки

Признаки: счётчики за бакет, окна 12ч/24ч/48ч/7д/30д, z-скор,
сезонность (час_бакета, день недели, месяц, день года), категории
`ид_объект_code`/`тип_датчика_code` и бакетный контекст объекта (`_об_б`).
Временной сплит — как в 08: train ≤ 2024, val = 2025, test = 2026.

In [ ]:
# Категориальные коды для бустинга
panel["ид_объект_code"] = panel["ид_объект"].astype("category").cat.codes
panel["тип_датчика_code"] = panel["тип_датчика"].astype("category").cat.codes

train, val, test = fe.split_by_time(panel)
drop_cols = ["ид_канала_данных", "бакет", "дата", "год_неделя", "аномально",
             "ид_объект", "тип_датчика"] + [f"цель_{n}" for n in fe.HORIZONS]
X_cols = [c for c in train.columns if c not in drop_cols]

print("Train:", train.shape, "| Val:", val.shape, "| Test:", test.shape)
print("Признаков:", len(X_cols))
print("\nДоля positive (train, вне кампаний):")
for n in fe.HORIZONS:
    print(f"  цель_{n}: {100 * train[f'цель_{n}'].mean():6.3f}%")

# 9.3 · Обучение моделей по всем горизонтам (CatBoost, CPU)

Для каждого горизонта обучается отдельная модель на ОДНИХ и тех же признаках
(меняется только целевая переменная). Early stopping по F1 на валидации,
оценка на тесте 2026.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             f1_score, precision_score as ps, recall_score as rs)

ITERS, LR, DEPTH, ES = 700, 0.05, 6, 300
rows = []
for name in fe.HORIZONS:
    y_col = f"цель_{name}"
    tr = train.dropna(subset=[y_col])
    va = val.dropna(subset=[y_col])
    te = test.dropna(subset=[y_col])
    m = CatBoostClassifier(iterations=ITERS, learning_rate=LR, depth=DEPTH,
                           random_seed=42, task_type="CPU", eval_metric="F1",
                           early_stopping_rounds=ES, verbose=100)
    m.fit(Pool(tr[X_cols], tr[y_col]), eval_set=Pool(va[X_cols], va[y_col]))
    yp = m.predict_proba(te[X_cols])[:, 1]
    yt = te[y_col].astype(int).values
    ap = average_precision_score(yt, yp)
    pr, rc, _ = precision_recall_curve(yt, yp)
    p_r05 = float(pr[rc >= 0.5].max()) if (rc >= 0.5).any() else float("nan")
    f1 = float(f1_score(yt, (yp >= 0.5).astype(int)))
    best = None
    for thr in np.round(np.arange(0.20, 0.99, 0.01), 3):
        ypr = (yp >= thr).astype(int)
        p_, r_ = ps(yt, ypr, zero_division=0), rs(yt, ypr)
        if p_ >= 0.7 and (best is None or r_ > best[2]):
            best = (thr, p_, r_)
    rows.append({
        "горизонт": name, "база,%": round(100 * yt.mean(), 2),
        "PR-AUC": round(ap, 4), "Prec@R>=.5": round(p_r05, 4), "F1@.5": round(f1, 4),
        "thr@P>=.7": (None if best is None else best[0]),
        "Prec@thr": (None if best is None else round(best[1], 3)),
        "Rec@thr": (None if best is None else round(best[2], 3)),
    })
    print(f"Горизонт {name}: PR-AUC={ap:.4f}, iters={m.get_best_iteration()}")

res = pd.DataFrame(rows)
print("\nРезультаты на тесте 2026 (без кампаний):")
print(res.to_string(index=False))

# Как падает PR-AUC с горизонтом
plt.figure(figsize=(8, 4.2))
plt.plot(res["горизонт"], res["PR-AUC"], marker="o", linewidth=2, color="C0")
plt.title("PR-AUC по горизонту прогноза (тест 2026)")
plt.ylabel("PR-AUC")
plt.grid(alpha=0.3)
plt.ylim(0, 1)
plt.show()

# 9.4 · Выводы

- Модель обучается отдельно на каждый горизонт (6/12/24/48ч) на **одних и тех же
  признаках** — меняется только целевая переменная («неисправность в ближайшие
  N бакетов»).
- Ожидаемый паттерн: **PR-AUC и Precision@Recall≥0.5 падают с горизонтом**
  (ближайшие 6ч предсказать проще, чем 48ч — сигнатура «охлаждается»).
- Горизонт 24ч — целевой по ТЗ (≥24ч); горизонт 48ч показывает, сколько
  точности мы теряем за дополнительный день.
- Для применения: диспетчер видит вероятность события в **каждом из горизонтов**
  (6ч = «вот-вот», 24ч = «сегодня», 48ч = «ближайшие двое суток»), порог
  выбирается по таблице `thr@P>=.7`.